# Cloud Transfer Notebook

Copy public Yandex Disk files into your Google Drive. Run the cells below in order using a CPU runtime.

Files are downloaded to the runtime's local disk in 1 MiB chunks, verified, and then copied to Google Drive. Allow enough free local space for one file. A GPU is unnecessary. The original files remain on Yandex Disk.


## 1. Install dependencies


In [ ]:
%pip install -q 'requests>=2.32.4,<3' 'tqdm>=4.67,<5'


## 2. Connect Google Drive

Choose the Google account that should receive the files.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. Choose your files

Each dictionary key names a destination subdirectory. Use a list to copy multiple links into the same subdirectory:

```python
yandex_links = {
    "Files": [
        "https://disk.yandex.ru/i/FIRST_PUBLIC_KEY",
        "https://disk.yandex.ru/d/SECOND_PUBLIC_KEY",
    ],
    "Archive": "https://disk.yandex.ru/d/ANOTHER_PUBLIC_KEY",
}
```

Single links are also supported. Do not repeat dictionary keys: Python keeps only the last value. Files retain their original names, and a folder's contents are copied into its named subdirectory. Keep `overwrite` disabled to preserve existing files; matching files are verified and skipped.


In [ ]:
destination = "/content/drive/MyDrive/cloud-import"
yandex_links = {}
overwrite = False


## 4. Load the transfer code

This notebook includes the transfer code and does not need any other project files.


In [ ]:
import hashlib
import logging
import os
import re
import shutil
import tempfile
from dataclasses import dataclass
from pathlib import Path
from urllib.parse import unquote, urlsplit

import requests
from tqdm.auto import tqdm

API_URL = "https://cloud-api.yandex.net/v1/disk/public/resources"
CHUNK_SIZE = 1024 * 1024
PAGE_SIZE = 100
API_TIMEOUT = (60, 60)
DOWNLOAD_TIMEOUT = (60, 60)
CACHE_DIR = Path(tempfile.gettempdir()) / "cloud-transfer-notebook"
LOGGER = logging.getLogger(__name__)


class MigrationError(Exception):
    pass


class IntegrityError(MigrationError):
    pass


@dataclass
class MigrationResult:
    downloaded: int = 0
    skipped: int = 0
    failed: int = 0


def parse_public_url(url: str) -> tuple[str, str]:
    parsed = urlsplit(url)
    parts = parsed.path.strip("/").split("/")
    if (
        parsed.scheme != "https"
        or parsed.netloc not in {"disk.yandex.ru", "disk.yandex.com", "yadi.sk"}
        or len(parts) < 2
        or parts[0] not in {"d", "i"}
        or not parts[1]
    ):
        raise ValueError("Use a public HTTPS link from Yandex Disk (/d/... or /i/...).")
    public_key = f"https://{parsed.netloc}/{parts[0]}/{parts[1]}"
    path = unquote("/" + "/".join(parts[2:])) if len(parts) > 2 else ""
    return public_key, path


def safe_child(parent: Path, name: str, root: Path) -> Path:
    if (
        not isinstance(name, str)
        or not name
        or name in {".", ".."}
        or any(char in name for char in "/\\\0")
    ):
        raise MigrationError(f"Unsafe file or directory name: {name!r}")
    child = parent / name
    if child.is_symlink() or not child.resolve().is_relative_to(root):
        raise MigrationError(f"Destination escapes the selected directory: {name!r}")
    return child


def prepare_jobs(links: dict, destination: str | Path) -> list[tuple[str, str]]:
    if not isinstance(links, dict) or not links:
        raise ValueError("Add your public links to yandex_links in step 3.")
    root = Path(destination).expanduser().resolve()
    jobs = []
    for folder_name, urls in links.items():
        safe_child(root, folder_name, root)
        if isinstance(urls, str):
            urls = [urls]
        if not isinstance(urls, list) or not urls:
            raise ValueError(
                f"Use a public link or a nonempty list of links for {folder_name!r}."
            )
        for url in urls:
            if not isinstance(url, str):
                raise ValueError(f"Each link in {folder_name!r} must be a string.")
            parse_public_url(url)
            jobs.append((folder_name, url))
    return jobs


def request_error(error: requests.RequestException) -> str:
    if error.response is not None:
        return f"HTTP {error.response.status_code}"
    return type(error).__name__


def get_json(session: requests.Session, endpoint: str, params: dict) -> dict:
    try:
        with session.get(endpoint, params=params, timeout=API_TIMEOUT) as response:
            response.raise_for_status()
            data = response.json()
        if not isinstance(data, dict) or "error" in data:
            raise MigrationError("Yandex Disk API returned an invalid response.")
        return data
    except requests.RequestException as error:
        raise MigrationError(
            f"Yandex Disk API request failed: {request_error(error)}"
        ) from error
    except ValueError as error:
        raise MigrationError("Yandex Disk API returned invalid JSON.") from error


def get_resource(
    session: requests.Session, public_key: str, path: str, offset: int = 0
) -> dict:
    params = {
        "public_key": public_key,
        "limit": PAGE_SIZE,
        "offset": offset,
        "sort": "name",
    }
    if path:
        params["path"] = path
    return get_json(session, API_URL, params)


def iter_children(
    session: requests.Session, public_key: str, path: str, first_page: dict
):
    page = first_page
    offset = 0
    while True:
        embedded = page.get("_embedded")
        if not isinstance(embedded, dict):
            raise MigrationError(
                "Yandex Disk API returned an invalid directory listing."
            )
        items = embedded.get("items")
        total = embedded.get("total")
        if not isinstance(items, list) or not isinstance(total, int) or total < 0:
            raise MigrationError(
                "Yandex Disk API returned an invalid directory listing."
            )
        if not items and offset < total:
            raise MigrationError(
                "Directory listing ended before all entries were returned."
            )
        for item in items:
            if not isinstance(item, dict) or not isinstance(item.get("path"), str):
                raise MigrationError(
                    "Yandex Disk API returned an invalid directory entry."
                )
            yield item
        offset += len(items)
        if offset >= total:
            return
        page = get_resource(session, public_key, path, offset)


def file_metadata(resource: dict) -> tuple[int, str | None, str | None]:
    size = resource.get("size")
    if type(size) is not int or size < 0:
        raise MigrationError("File metadata is missing a valid size.")
    for algorithm in ("sha256", "md5"):
        expected = resource.get(algorithm)
        if expected is not None:
            length = 64 if algorithm == "sha256" else 32
            if (
                not isinstance(expected, str)
                or len(expected) != length
                or any(char not in "0123456789abcdefABCDEF" for char in expected)
            ):
                raise MigrationError(
                    f"File metadata has an invalid {algorithm} checksum."
                )
            return size, algorithm, expected.lower()
    return size, None, None


def verify_file(path: Path, resource: dict, *, require_checksum: bool = False) -> bool:
    size, algorithm, expected = file_metadata(resource)
    if path.stat().st_size != size:
        return False
    if algorithm is None:
        return not require_checksum
    digest = hashlib.new(algorithm, usedforsecurity=False)
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(CHUNK_SIZE), b""):
            digest.update(chunk)
    return digest.hexdigest() == expected


def download_file(
    session: requests.Session,
    public_key: str,
    resource: dict,
    target: Path,
    *,
    overwrite: bool,
) -> bool:
    """Return True for a new download and False for an already verified file."""
    expected_size, algorithm, expected_checksum = file_metadata(resource)
    if target.exists():
        if not target.is_file():
            raise MigrationError("Destination is not a regular file.")
        if not overwrite:
            if verify_file(target, resource, require_checksum=True):
                return False
            raise MigrationError(
                "Existing file differs or cannot be verified; set overwrite = True to replace it."
            )

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    identity = repr((
        public_key, resource["path"], expected_size, algorithm,
        expected_checksum, resource.get("modified"),
    ))
    cache_key = hashlib.sha256(identity.encode()).hexdigest()
    partial = CACHE_DIR / f"{cache_key}.part"
    params = {"public_key": public_key, "path": resource["path"]}

    try:
        offset = partial.stat().st_size if partial.exists() else 0
        if offset > expected_size:
            partial.unlink()
            offset = 0
        if offset < expected_size or not partial.exists():
            if shutil.disk_usage(CACHE_DIR).free < expected_size - offset:
                raise MigrationError(
                    "Not enough local disk space for this file. "
                    "Free space in the Colab runtime and rerun the copying cell."
                )
            link = get_json(session, f"{API_URL}/download", params).get("href")
            if not isinstance(link, str) or urlsplit(link).scheme != "https":
                raise MigrationError(
                    "Yandex Disk did not return an HTTPS download link."
                )
            headers = {"Accept-Encoding": "identity"}
            if offset:
                headers["Range"] = f"bytes={offset}-"
            with session.get(
                link, headers=headers, stream=True, timeout=DOWNLOAD_TIMEOUT
            ) as response:
                if response.status_code == 416:
                    raise IntegrityError("Server rejected the resume offset.")
                response.raise_for_status()
                if response.status_code == 206:
                    content_range = re.fullmatch(
                        r"bytes (\d+)-(\d+)/(\d+)",
                        response.headers.get("Content-Range", ""),
                    )
                    if (
                        content_range is None
                        or int(content_range[1]) != offset
                        or int(content_range[3]) != expected_size
                        or not offset <= int(content_range[2]) < expected_size
                    ):
                        raise IntegrityError("Invalid download Content-Range.")
                    mode = "ab" if offset else "wb"
                elif response.status_code == 200:
                    if offset:
                        LOGGER.warning(
                            "%s: server does not support this resume request; restarting",
                            target.name,
                        )
                    offset = 0
                    mode = "wb"
                else:
                    raise MigrationError(
                        f"Unexpected download status: HTTP {response.status_code}"
                    )
                encoding = response.headers.get("Content-Encoding", "identity")
                if encoding.lower() not in {"", "identity"}:
                    raise MigrationError("Cannot resume an encoded download response.")
                with partial.open(mode) as output, tqdm(
                    desc=target.name,
                    total=expected_size,
                    initial=offset,
                    unit="B",
                    unit_scale=True,
                    unit_divisor=1024,
                    leave=False,
                ) as progress:
                    for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            output.write(chunk)
                            progress.update(len(chunk))
        if partial.stat().st_size < expected_size:
            raise requests.ConnectionError("Download ended before the expected size.")
        if not verify_file(partial, resource):
            raise IntegrityError(
                "Downloaded file does not match its size or checksum."
            )
    except requests.RequestException as error:
        saved = partial.stat().st_size if partial.exists() else 0
        raise MigrationError(
            f"Download failed: {request_error(error)}; "
            f"saved {saved}/{expected_size} bytes. "
            "Rerun the copying cell in this runtime to retry."
        ) from error
    except IntegrityError:
        partial.unlink(missing_ok=True)
        raise

    temporary = None
    try:
        LOGGER.info("Copying verified file to Google Drive: %s", target.name)
        with tempfile.NamedTemporaryFile(
            dir=target.parent, prefix=".yadisk-", suffix=".part", delete=False
        ) as output:
            temporary = Path(output.name)
            with partial.open("rb") as downloaded:
                shutil.copyfileobj(downloaded, output, length=CHUNK_SIZE)
        if not verify_file(temporary, resource):
            raise IntegrityError("Google Drive copy failed verification; local copy retained.")
        if target.is_symlink() or (target.exists() and not overwrite):
            raise MigrationError(
                "Destination appeared during download; refusing to replace it."
            )
        os.replace(temporary, target)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
    partial.unlink(missing_ok=True)
    return True


def migrate(
    public_url: str,
    destination: str | Path,
    *,
    folder_name: str | None = None,
    overwrite: bool = False,
) -> MigrationResult:
    public_key, path = parse_public_url(public_url)
    root = Path(destination).expanduser().resolve()
    result = MigrationResult()
    with requests.Session() as session:
        session.headers.update({"User-Agent": "cloud-transfer-notebook/1.0"})
        resource = get_resource(session, public_key, path)
        root.mkdir(parents=True, exist_ok=True)
        if folder_name is not None:
            root = safe_child(root, folder_name, root)
            root.mkdir(exist_ok=True)
        pending = [(resource, root, path)]
        while pending:
            current, parent, current_path = pending.pop()
            try:
                resource_type = current.get("type")
                if resource_type == "file":
                    target = safe_child(parent, current["name"], root)
                    downloaded = download_file(
                        session, public_key, current, target, overwrite=overwrite
                    )
                    if downloaded:
                        result.downloaded += 1
                    else:
                        result.skipped += 1
                    LOGGER.info(
                        "%s: %s",
                        "Downloaded" if downloaded else "Verified, skipped",
                        target,
                    )
                elif resource_type == "dir":
                    if current is resource:
                        target = parent
                        first_page = current
                    else:
                        target = safe_child(parent, current["name"], root)
                        first_page = get_resource(session, public_key, current_path)
                    target.mkdir(exist_ok=True)
                    for child in iter_children(
                        session, public_key, current_path, first_page
                    ):
                        pending.append((child, target, child["path"]))
                else:
                    raise MigrationError(
                        f"Unsupported resource type: {resource_type!r}"
                    )
            except (MigrationError, OSError, KeyError) as error:
                result.failed += 1
                LOGGER.error("Failed %s: %s", current.get("name", "resource"), error)
    return result


## 5. Start copying

The report shows downloaded files, existing files verified and skipped, and errors.

Each API request and file download is attempted once, without automatic retries. Connection and read timeouts are both 60 seconds. The read timeout limits waiting for data, not the total download duration. Failed files are reported, and copying continues with the next file.

Partial downloads are kept on the runtime's local disk and resumed when the server supports range requests. If the server ignores a resume request, the download starts from the beginning. Invalid range responses and failed integrity checks are reported without an automatic second attempt.

Rerun this cell to retry failed transfers in the same runtime. Resetting or deleting the runtime removes the local partial files. A verified local download is retained if copying it to Google Drive fails.


In [ ]:
if not Path("/content/drive/MyDrive").is_dir():
    raise ValueError("Connect Google Drive in step 2 before starting the transfer.")
jobs = prepare_jobs(yandex_links, destination)

LOGGER.setLevel(logging.INFO)
if not LOGGER.handlers:
    LOGGER.addHandler(logging.StreamHandler())
LOGGER.propagate = False
total = MigrationResult()
for folder_name, public_url in jobs:
    print(f"\nCopying into: {folder_name} — {public_url}")
    try:
        result = migrate(
            public_url, destination, folder_name=folder_name, overwrite=overwrite
        )
    except (MigrationError, OSError, ValueError) as error:
        LOGGER.error("Failed %s: %s", folder_name, error)
        result = MigrationResult(failed=1)
    total.downloaded += result.downloaded
    total.skipped += result.skipped
    total.failed += result.failed
    print(f"Downloaded: {result.downloaded}; verified, skipped: {result.skipped}; errors: {result.failed}")

print(f"\nTotal — downloaded: {total.downloaded}; verified, skipped: {total.skipped}; errors: {total.failed}")
